In [1]:
import pickle
import os
import pandas as pd
from collections import Counter, defaultdict
import ast

In [1]:
pred_tuples_llm_dict = pickle.load(open('property_test_results_deepseek_2.pkl','rb'))
gold_disco_tuples_test = pickle.load(open('test_gold_tuples.pkl', 'rb'))
# # print(gold_disco_tuples_test[:10])
# # print(type(pred_tuples_llm_dict))

In [15]:
# pred_tuples_llm_dict

In [16]:
def process_dict_to_list_of_tuples(data_dict):
    result = []
    
    for value in data_dict.values():
        # Skip empty lists
        if value == '[]' or value == []:
            continue
            
        # Skip error messages
        if isinstance(value, str) and value.startswith('Error:'):
            continue
            
        # The value is already a list of lists of tuples, so we can directly process it
        if isinstance(value, list):
            for sublist in value:
                for item in sublist:
                    result.append(item)
    
    return result

In [17]:
pred_disco_tuples_test = process_dict_to_list_of_tuples(pred_tuples_llm_dict)
# pred_disco_tuples_test = pickle.load(open('test_pred_tuples_no_checker.pkl','rb'))
print(len(gold_disco_tuples_test), len(pred_disco_tuples_test))

3520 3402


In [18]:
# pred_tuples_llm_dict

In [19]:
def match_tuple(p, g):
    
    if p[0].count('_') == 4:
        p0 = p[0][:p[0].rindex('_')]
    else:
        p0 = p[0]
        
    if g[0].count('_') == 4:
        g0 = g[0][:g[0].rindex('_')]
    else:
        g0 = g[0]
        
#     p0 = p[0][:p[0].rindex('_')]
#     g0 = g[0][:g[0].rindex('_')]
#     print(f'{p[0]}  ---->  {p0}')
#     print(f'{g[0]}  ---->  {g0}')
#     print()
#     if p0 == g0 and p[0]!=g[0]:
#         print(p)
#         print(g)
#         print()
    try:
        return p0 == g0 and p[1] == g[1] and round(abs(p[2] - g[2]),2)==0 and p[3] == g[3]
    except TypeError:
        print('Check for type mismatch')
        print(p)
        print(g)
        return False

def get_tuples_metrics(gold_tuples, pred_tuples):
    global mismatch_train, mismatch_val, mismatch_test
    prec = 0
    for p in pred_tuples:
        for g in gold_tuples:
            if match_tuple(p, g):
                prec += 1
                break
        # if p in gold_tuples:
        #     prec += 1
    if len(pred_tuples) > 0:
        prec /= len(pred_tuples)
    else:
        prec = 0.0
    rec = 0
    for g in gold_tuples:
        for p in pred_tuples:
            if match_tuple(p, g):
                rec += 1
                break
        # if g in pred_tuples:
        #     rec += 1
#     new_gold_tup = [tup for tup in gold_tuples if tup[-2]!='']
    rec /= len(gold_tuples)
    fscore = 2 * prec * rec / (prec + rec) if (prec + rec > 0) else 0.0

    return {'precision': round(prec,4), 'recall': round(rec,4), 'fscore': round(fscore,4)}

In [20]:
tuple_metrics = get_tuples_metrics(gold_disco_tuples_test, pred_disco_tuples_test)
print(tuple_metrics)

{'precision': 0.7043, 'recall': 0.6807, 'fscore': 0.6923}


In [2]:
# print(gold_disco_tuples_test[:20])
# print()
# print(pred_disco_tuples_test[:20])

In [23]:
prop_names = ['Density', 'Glass transition temperature', 'Refractive index', 'Abbe value', "Young's modulus", 'Shear modulus', 'Vickers hardness', 'Poisson ratio', 'Fracture toughness', 'Crystallization temp', 'Melting temp', 'Electric conductivity', 'Dielectric constant', 'Softening Point (Temperature)', 'Annealing Point (Temperature)', 'Thermal expansion coefficient', 'Liquidus temperature', 'Bulk modulus', 'Activation energy']

In [24]:
def get_property_metrics(gold_tuples, pred_tuples, prop_names):
    
    results = {}
    for prop in prop_names:
        if prop == 'Dielectric constant':
            continue
        # Filter tuples for the current property
        gold_filtered = [t for t in gold_tuples if t[1] == prop]
        pred_filtered = [t for t in pred_tuples if t[1] == prop]
        
        # Calculate metrics for the filtered tuples
        results[prop] = get_tuples_metrics(gold_filtered, pred_filtered)
    
    return results

In [26]:
prop_metrics = get_property_metrics(gold_disco_tuples_test, pred_disco_tuples_test, prop_names)
print(prop_metrics)

{'Density': {'precision': 0.7265, 'recall': 0.7396, 'fscore': 0.733}, 'Glass transition temperature': {'precision': 0.7202, 'recall': 0.7728, 'fscore': 0.7456}, 'Refractive index': {'precision': 0.8472, 'recall': 0.8291, 'fscore': 0.838}, 'Abbe value': {'precision': 1.0, 'recall': 1.0, 'fscore': 1.0}, "Young's modulus": {'precision': 0.6171, 'recall': 0.6968, 'fscore': 0.6545}, 'Shear modulus': {'precision': 0.5164, 'recall': 0.5, 'fscore': 0.5081}, 'Vickers hardness': {'precision': 0.5385, 'recall': 0.4884, 'fscore': 0.5122}, 'Poisson ratio': {'precision': 0.8519, 'recall': 0.6216, 'fscore': 0.7187}, 'Fracture toughness': {'precision': 0.1786, 'recall': 0.1786, 'fscore': 0.1786}, 'Crystallization temp': {'precision': 0.9017, 'recall': 0.5009, 'fscore': 0.6441}, 'Melting temp': {'precision': 0.9658, 'recall': 0.8248, 'fscore': 0.8898}, 'Electric conductivity': {'precision': 0.8841, 'recall': 0.9242, 'fscore': 0.9037}, 'Softening Point (Temperature)': {'precision': 0.825, 'recall': 0.50